In [5]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 40)
RANDOM_STATE = 42

# LOAD
df = pd.read_csv("yahoo_screener_full.csv", low_memory=False)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")


# DROP REDUNDANT / USELESS COLUMNS
# Drop .fmt and .longFmt mirrors — we keep .raw (numeric).
fmt_cols = [c for c in df.columns if c.endswith(".fmt") or c.endswith(".longFmt")]
df = df.drop(columns=fmt_cols)
print(f"Dropped {len(fmt_cols)} formatted (string) mirror columns")

# Drop constant columns
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=constant_cols)
print(f"Dropped {len(constant_cols)} constant columns")

# Drop metadata / identifiers that add no analytical value
meta_cols = [
    "messageBoardId", "quoteSourceName", "language", "region", "market",
    "exchangeTimezoneName", "exchangeTimezoneShortName", "gmtOffSetMilliseconds",
    "sourceInterval", "exchangeDataDelayedBy", "priceHint",
    "hasPrePostMarketData", "cryptoTradeable", "tradeable", "triggerable",
    "customPriceAlertConfidence", "firstTradeDateMilliseconds",
    "corporateActions", "prevName", "nameChangeDate", "displayName",
    "quoteType", "typeDisp", "exchange", "fullExchangeName", "currency",
    "marketState", "ipoExpectedDate",
]
df = df.drop(columns=[c for c in meta_cols if c in df.columns], errors="ignore")
print(f"Dropped {len(meta_cols)} metadata columns")

# HANDLE MISSING DATA
# Drop rows missing the essential identifier
df = df.dropna(subset=["symbol"]).reset_index(drop=True)

# Drop columns whose missingness is too high (>60%)
missing_ratio = df.isnull().mean()
high_missing = missing_ratio[missing_ratio > 0.60].index.tolist()
df = df.drop(columns=high_missing)
print(f" Dropped {len(high_missing)} columns with >60% missing values")
print(f"Remaining NaN per column (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))


#  PARSE / FEATURE-ENGINEER SPECIAL COLUMNS
# averageAnalystRating → numeric score + ordinal label
if "averageAnalystRating" in df.columns:
    df["analyst_score"] = (
        df["averageAnalystRating"].astype(str).str.extract(r"([\d.]+)").astype(float)
    )
    df["analyst_label"] = (
        df["averageAnalystRating"].astype(str).str.extract(r"- (.+)$")[0]
    )
    df = df.drop(columns=["averageAnalystRating"])
    print("Split 'averageAnalystRating' → 'analyst_score' + 'analyst_label'")

#  Coerce boolean-like strings to real booleans
for col in df.columns:
    if df[col].dtype == object:
        uniq = set(df[col].dropna().astype(str).str.lower().unique())
        if uniq.issubset({"true", "false"}):
            df[col] = df[col].astype(str).str.lower().map({"true": True, "false": False})

# Ensure every *.raw numeric field is truly numeric (some are strings)
for col in df.columns:
    if col.endswith(".raw") or col in {
        "regularMarketPrice", "regularMarketChange", "regularMarketChangePercent",
        "regularMarketVolume", "marketCap", "regularMarketPreviousClose",
    }:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# OUTLIER DETECTION & REMOVAL
# Use IQR on key price/volume/market-cap features. We WINSORIZE instead of
# hard-dropping so we don't lose precious rows (only ~30 tickers here).
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

key_outlier_cols = [c for c in numeric_cols if any(k in c for k in
                    ("marketCap", "regularMarketPrice", "regularMarketVolume",
                     "regularMarketChangePercent"))]

def winsorize(series, lower_q=0.01, upper_q=0.99):
    lo, hi = series.quantile([lower_q, upper_q])
    return series.clip(lower=lo, upper=hi)

for col in key_outlier_cols:
    df[col] = winsorize(df[col])

print(f"Winsorized {len(key_outlier_cols)} key numeric columns at 1–99%")
print(f"(rows retained: {len(df)})")

# FEATURE LISTS (no overlap, no all-NaN columns)

symbols = df["symbol"].copy()
feature_df = df.drop(columns=["symbol"])

categorical_features = feature_df.select_dtypes(
    include=["object", "bool", "category"]
).columns.tolist()

numeric_features = [
    c for c in feature_df.columns
    if c not in categorical_features
    and pd.api.types.is_numeric_dtype(feature_df[c])
]

# Drop all-NaN columns BEFORE the pipeline
numeric_features = [c for c in numeric_features if feature_df[c].notna().any()]
categorical_features = [c for c in categorical_features if feature_df[c].notna().any()]

assert set(numeric_features).isdisjoint(categorical_features)

print(f"🔢 Numeric     : {len(numeric_features)}")
print(f"🔤 Categorical : {len(categorical_features)}")


# PIPELINE
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

X = preprocessor.fit_transform(feature_df)
all_feature_names = preprocessor.get_feature_names_out().tolist()

assert X.shape[1] == len(all_feature_names), \
    f"Mismatch: X has {X.shape[1]} cols, names has {len(all_feature_names)}"

X_df = pd.DataFrame(X, columns=all_feature_names, index=feature_df.index)
X_df.insert(0, "symbol", symbols.values)
print(f"Processed matrix: {X_df.shape[0]} rows x {X_df.shape[1]} columns")

#  LABEL-ENCODED VARIANT
label_df = feature_df.copy()
for col in categorical_features:
    le = LabelEncoder()
    label_df[col] = le.fit_transform(label_df[col].astype(str).fillna("unknown"))
label_df[numeric_features] = label_df[numeric_features].fillna(
    label_df[numeric_features].median()
)
label_df.insert(0, "symbol", symbols.values)

# SAVE
X_df.to_csv("yahoo_screener_processed.csv", index=False)
label_df.to_csv("yahoo_screener_label_encoded.csv", index=False)
df.to_csv("yahoo_screener_clean.csv", index=False)
print("Saved all 3 CSVs")

# SANITY CHECK (fixed to use prefixed names)
print("Preview of processed matrix:")
print(X_df.iloc[:5, :8])

numeric_out = [c for c in X_df.columns if c.startswith("num__")]
print(f"Summary statistics (first 8 numeric features of {len(numeric_out)}):")
print(X_df[numeric_out[:8]].describe().T[["mean", "std", "min", "max"]])

print("Task 2 of Level 1 completed by Omar")

Loaded: 400 rows × 168 columns
Dropped 74 formatted (string) mirror columns
Dropped 15 constant columns
Dropped 28 metadata columns
 Dropped 0 columns with >60% missing values
Remaining NaN per column (top 10):
dividendRate.raw              224
dividendYield.raw             224
trailingPE.raw                176
dividendDate.raw              128
lastCloseTevEbitLtm.raw        80
preMarketChangePercent.raw     32
preMarketTime.raw              32
preMarketPrice.raw             32
preMarketChange.raw            32
averageAnalystRating           32
dtype: int64
Split 'averageAnalystRating' → 'analyst_score' + 'analyst_label'
Winsorized 4 key numeric columns at 1–99%
(rows retained: 400)
🔢 Numeric     : 59
🔤 Categorical : 5
Processed matrix: 400 rows x 119 columns
Saved all 3 CSVs
Preview of processed matrix:
  symbol  num__twoHundredDayAverageChangePercent.raw  \
0     NU                                   -0.109612   
1   INTC                                    2.080132   
2   CTVA        